In [ ]:
# ── BFCL paper experiments: config ──
# Reads the runs written by scripts/bfcl_capacity_sweep.sh and prints the
# median FTR and E2E table.
SYSNAME = 'Sutradhara'
SYSNAME_SHORT = 'SD'

import os, glob, csv, re
import numpy as np

def _find_repo_root():
    env = os.environ.get('SUTRADHARA_REPO')
    if env:
        return os.path.abspath(env)
    p = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(p, 'pyproject.toml')):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return os.path.abspath(os.getcwd())
        p = parent

REPO_ROOT = _find_repo_root()
BASE = os.environ.get('SUTRADHARA_EXPERIMENTS', os.path.join(REPO_ROOT, 'experiments'))
PLOT_DIR = os.path.join(REPO_ROOT, 'analysis', 'paper_plots')
os.makedirs(PLOT_DIR, exist_ok=True)

# ── BFCL run configuration (must match scripts/bfcl_capacity_sweep.sh) ──
TRACE    = os.environ.get(
    'BFCL_TRACE_NAME',
    'bfcl_trace')
MEM      = '065'
SEED     = 42
# 'latest' or 'all
RUN_DIR_MODE = os.environ.get('BFCL_RUN_DIR_MODE', 'all').lower()
if RUN_DIR_MODE not in {'latest', 'all'}:
    raise ValueError("BFCL_RUN_DIR_MODE must be 'latest' or 'all'")
# QPS points are discovered from disk, so adding sweep points needs no edit here.
def _discover_qps():
    pat = os.path.join(BASE, TRACE, f'chunk256_qps*_mem{MEM}')
    qs = []
    for d in glob.glob(pat):
        m = re.search(r'chunk256_qps([0-9.]+)_mem', d)
        if m:
            qs.append(float(m.group(1)))
    return sorted(set(qs))

# Cap the sweep: past ~4 QPS the 56-request trace arrives in under 10s, so a
# run measures a burst rather than a load level.
QPS_MAX = float(os.environ.get('BFCL_QPS_MAX', 4.0))
QPS_LIST = [q for q in _discover_qps() if q <= QPS_MAX]
PICK_QPS = 1.5          # ablation / breakdown / KV analysis load point
TAG_BL, TAG_SD = 'baseline', 'ds_kv'
TAG_LRU, TAG_PRI = 'et', 'ds_kv_et'

print(f'REPO_ROOT  {REPO_ROOT}')
print(f'BASE       {BASE}')
print(f'TRACE      {TRACE}')
print(f'QPS_LIST   {QPS_LIST}')
print(f'RUN_DIR_MODE {RUN_DIR_MODE}')


def run_dir(qps, tag, seed=SEED, trace=None, required=True,
            selection='latest', metrics_name='request_metrics.csv'):
    """Selected run dir(s) for (qps, tag). Layout written by run_experiments.py:
    experiments/{trace}/chunk256_qps{qps}_mem{MEM}/{tag}/seed{seed}/{ts}/"""
    if selection not in {'latest', 'all'}:
        raise ValueError("selection must be 'latest' or 'all'")
    parent = os.path.join(BASE, trace or TRACE, f'chunk256_qps{qps}_mem{MEM}', tag)
    if seed is not None:
        parent = os.path.join(parent, f'seed{seed}')
    runs = sorted(
        d for d in glob.glob(parent + '/*')
        if os.path.isfile(os.path.join(d, 'metrics', metrics_name))
    )
    if runs:
        return runs if selection == 'all' else runs[-1]
    if required:
        raise FileNotFoundError(f'No run with metrics under {parent}')
    return None


def req_rows(qps, tag, seed=SEED, name='request_metrics.csv', required=True):
    d = run_dir(qps, tag, seed, required=required, metrics_name=name)
    if d is None:
        return None
    return req_rows_from_dir(d, name)


def req_rows_from_dir(run_path, name='request_metrics.csv'):
    return list(csv.DictReader(open(os.path.join(run_path, 'metrics', name))))


def have(qps, tag, seed=SEED):
    return run_dir(qps, tag, seed, required=False) is not None


def pctls(rows, col, scale=1000.0):
    v = np.array([float(r[col]) / scale for r in rows])
    return {'mean': v.mean(), 'p50': np.median(v),
            'p90': np.percentile(v, 90), 'p99': np.percentile(v, 99)}

In [ ]:
# BFCL: median FTR / E2E, baseline vs Sutradhara (DS+KV)
def print_latency_table(run_index=None):
    label = 'latest run' if run_index is None else f'run set {run_index + 1}'
    print(f'\nBFCL median latency — {label}')
    print(f"{'':>5}{'FTR p50':>23}{'E2E p50':>23}")
    print(f"{'QPS':>5}" + f"{'BL':>7}{'SD':>7}{'gain':>9}" * 2)
    rows_printed = 0
    for q in QPS_LIST:
        if run_index is None:
            bl_dir = run_dir(q, TAG_BL, required=False)
            sd_dir = run_dir(q, TAG_SD, required=False)
        else:
            bl_dirs = run_dir(
                q, TAG_BL, required=False, selection='all'
            ) or []
            sd_dirs = run_dir(
                q, TAG_SD, required=False, selection='all'
            ) or []
            if run_index >= min(len(bl_dirs), len(sd_dirs)):
                continue
            bl_dir, sd_dir = bl_dirs[run_index], sd_dirs[run_index]
        if bl_dir is None or sd_dir is None:
            continue

        bl = req_rows_from_dir(bl_dir)
        sd = req_rows_from_dir(sd_dir)
        bf = pctls(bl, 'first_token_render_ms')
        sf = pctls(sd, 'first_token_render_ms')
        be = pctls(bl, 'e2e_time_ms')
        se = pctls(sd, 'e2e_time_ms')
        line = f"{q:>5}"
        for b, s in ((bf, sf), (be, se)):
            baseline_value, sd_value = b['p50'], s['p50']
            line += (
                f"{baseline_value:>7.2f}{sd_value:>7.2f}"
                f"{(baseline_value - sd_value) / baseline_value * 100:>+8.1f}%"
            )
        print(line)
        rows_printed += 1
    if rows_printed == 0:
        print('  No paired baseline/SD runs available')


if RUN_DIR_MODE == 'latest':
    print_latency_table()
else:
    paired_run_counts = []
    for q in QPS_LIST:
        bl_dirs = run_dir(q, TAG_BL, required=False, selection='all') or []
        sd_dirs = run_dir(q, TAG_SD, required=False, selection='all') or []
        paired_run_counts.append(min(len(bl_dirs), len(sd_dirs)))
    for index in range(max(paired_run_counts, default=0)):
        print_latency_table(index)


In [ ]:
# Paper-style median/P90 QPS vs latency with variability across timestamped runs
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 9, 'axes.labelsize': 10,
    'axes.titlesize': 10, 'legend.fontsize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'lines.linewidth': 1.0,
})

C_BL, C_SD, C_ARROW = '#FF2C00', '#0C5DA5', '#009E73'


def timestamp_run_percentiles(qps, tag, seed, column, percentile):
    parent = os.path.join(
        BASE, TRACE, f'chunk256_qps{qps}_mem{MEM}', tag, f'seed{seed}'
    )
    run_dirs = sorted(
        path for path in glob.glob(os.path.join(parent, '*'))
        if os.path.isfile(os.path.join(path, 'metrics', 'request_metrics.csv'))
    )
    return np.asarray([
        pctls(
            list(csv.DictReader(open(os.path.join(path, 'metrics', 'request_metrics.csv')))),
            column,
        )[percentile]
        for path in run_dirs
    ])


def aggregate_timestamp_runs(qps_values, tag, seed, column, percentile):
    available_qps, means, stds, counts = [], [], [], []
    for qps in qps_values:
        values = timestamp_run_percentiles(qps, tag, seed, column, percentile)
        if values.size == 0:
            print(f'Skipping QPS={qps} for {tag}: no timestamped seed{seed} runs')
            continue
        available_qps.append(qps)
        means.append(values.mean())
        stds.append(values.std() if values.size > 1 else 0.0)
        counts.append(values.size)
    return (
        np.asarray(available_qps, dtype=float), np.asarray(means),
        np.asarray(stds), np.asarray(counts),
    )


def plot_qps_vs_latency_with_error_bands(
        qps_values, tag1, tag2, pick_qps=3.0, seed=SEED):
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.5), sharey=True)
    plot_configs = [
        ('first_token_render_ms', 'Median FTR (ms)', 'FTR p50', 'p50'),
        ('e2e_time_ms', 'Median E2E (ms)', 'E2E p50', 'p50'),
        ('first_token_render_ms', 'P90 FTR (ms)', 'FTR p90', 'p90'),
        ('e2e_time_ms', 'P90 E2E (ms)', 'E2E p90', 'p90'),
    ]

    for ax, (column, xlabel, short_label, percentile) in zip(
            axes.flat, plot_configs):
        bl_qps, bl_m, bl_s, bl_n = aggregate_timestamp_runs(
            qps_values, tag1, seed, column, percentile
        )
        sd_qps, sd_m, sd_s, sd_n = aggregate_timestamp_runs(
            qps_values, tag2, seed, column, percentile
        )
        qps_arr = np.intersect1d(bl_qps, sd_qps)
        if qps_arr.size == 0:
            raise FileNotFoundError(
                f'No QPS points have both {tag1} and {tag2} timestamped runs'
            )
        bl_indices = [int(np.flatnonzero(np.isclose(bl_qps, q))[0]) for q in qps_arr]
        sd_indices = [int(np.flatnonzero(np.isclose(sd_qps, q))[0]) for q in qps_arr]
        bl_m, bl_s, bl_n = bl_m[bl_indices], bl_s[bl_indices], bl_n[bl_indices]
        sd_m, sd_s, sd_n = sd_m[sd_indices], sd_s[sd_indices], sd_n[sd_indices]
        selected_index = int(np.argmin(np.abs(qps_arr - pick_qps)))
        selected_qps = float(qps_arr[selected_index])
        if not np.isclose(selected_qps, pick_qps):
            print(
                f'Marker QPS={pick_qps} is unavailable; using nearest '
                f'common QPS={selected_qps}'
            )

        ax.fill_betweenx(qps_arr, bl_m - bl_s, bl_m + bl_s, color=C_BL, alpha=0.15)
        ax.fill_betweenx(qps_arr, sd_m - sd_s, sd_m + sd_s, color=C_SD, alpha=0.15)
        ax.plot(bl_m, qps_arr, 'o--', color=C_BL, markersize=3.5, label='Baseline')
        ax.plot(sd_m, qps_arr, 'o-', color=C_SD, markersize=3.5, label=SYSNAME)
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.3)

        selected_latency = sd_m[selected_index]
        if not bl_m.min() <= selected_latency <= bl_m.max():
            raise ValueError(
                f'Selected {tag2} {short_label}={selected_latency:.2f} ms is outside '
                f'the {tag1} range [{bl_m.min():.2f}, {bl_m.max():.2f}] ms'
            )
        bl_order = np.argsort(bl_m)
        baseline_qps = float(np.interp(
            selected_latency, bl_m[bl_order], qps_arr[bl_order]
        ))
        qps_increase_pct = (selected_qps - baseline_qps) / baseline_qps * 100
        y_top = max(selected_qps, baseline_qps)
        y_bottom = min(selected_qps, baseline_qps)
        y_pad = (y_top - y_bottom) * 0.03
        ax.scatter(
            [selected_latency, selected_latency], [baseline_qps, selected_qps],
            color='black', s=10, zorder=4,
        )
        ax.annotate(
            '', xy=(selected_latency, y_top - y_pad),
            xytext=(selected_latency, y_bottom + y_pad),
            arrowprops=dict(arrowstyle='<->', color=C_ARROW, lw=0.8),
        )
        ax.annotate(
            f'{qps_increase_pct:+.0f}%',
            xy=(selected_latency, (y_top + y_bottom) / 2), xytext=(6, 0),
            textcoords='offset points', color=C_ARROW, fontsize=8,
            fontweight='bold', va='center',
        )

        max_latency_reduction = np.max((bl_m - sd_m) / bl_m * 100)
        ax.text(
            0.97, 0.04, f'Max {short_label} reduction: {max_latency_reduction:.0f}%',
            transform=ax.transAxes, fontsize=7, ha='right', va='bottom',
        )
        print(
            f'{short_label}: {tag2} latency={selected_latency:.2f} ms at '
            f'{selected_qps:.2f} QPS; {tag1}={baseline_qps:.2f} QPS; '
            f'QPS increase={qps_increase_pct:+.1f}%'
        )
        print(
            f'  replicate counts by QPS: {tag1}={bl_n.tolist()}, '
            f'{tag2}={sd_n.tolist()}'
        )

    axes[0, 0].set_ylabel('Load (QPS)')
    axes[1, 0].set_ylabel('Load (QPS)')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, frameon=False, ncol=2,
        loc='lower center', bbox_to_anchor=(0.5, -0.05),
    )
    plt.tight_layout(pad=0.5, w_pad=0.8, h_pad=0.8)
    save_path = os.path.join(PLOT_DIR, f'qps_vs_latency_error_bands_{tag1}_vs_{tag2}.pdf')
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved to {save_path}')
    plt.show()


plot_qps_vs_latency_with_error_bands(QPS_LIST, TAG_BL, TAG_SD, pick_qps=3.0)


## Per-request FTR latency breakdown
Compares the five shared requests with the largest FTR reduction from baseline to DS+KV. The displayed breakdown contains tool, prefill, and decode time. You can change the QPS point for which this breakdown is shown - set to 1.5 by default. Uses the latest run directory for this plot.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

C_TOOL, C_PREF, C_DEC = '#D95F02', '#1B9E77', '#BFBFBF'


def request_breakdown(row):
    prefill = float(row['total_prefill_time_ms']) / 1000
    decode = float(row['total_decode_time_ms']) / 1000
    #scheduling = float(row['total_scheduling_delay_ms']) / 1000
    ftr = float(row['first_token_render_ms']) / 1000
    tool = max(0, ftr - prefill - decode)
    return tool, prefill, decode


def plot_request_breakdown(qps, baseline_tag, sd_tag, title, save_name, n=5):
    baseline_rows = req_rows(qps, baseline_tag)
    sd_by_id = {row['request_id']: row for row in req_rows(qps, sd_tag)}

    records = []
    for baseline_row in baseline_rows:
        request_id = baseline_row['request_id']
        if request_id not in sd_by_id:
            continue
        sd_row = sd_by_id[request_id]
        baseline_ftr = float(baseline_row['first_token_render_ms']) / 1000
        sd_ftr = float(sd_row['first_token_render_ms']) / 1000
        ftr_reduction = baseline_ftr - sd_ftr
        if ftr_reduction <= 0:
            continue
        baseline_parts = request_breakdown(baseline_row)
        sd_parts = request_breakdown(sd_row)
        baseline_height = sum(baseline_parts)
        sd_height = sum(sd_parts)
        bar_reduction_pct = ((baseline_height - sd_height) / baseline_height * 100
                             if baseline_height else 0)
        records.append((
            request_id,
            bar_reduction_pct,
            baseline_parts,
            sd_parts,
            ftr_reduction,
        ))

    records.sort(key=lambda record: record[4], reverse=True)
    picked = records[:n]
    if not picked:
        raise ValueError(f'No matching requests for {baseline_tag} and {sd_tag} at QPS {qps}')

    fig, axes = plt.subplots(1, len(picked), figsize=(6.2, 2.6), sharey=True)
    if len(picked) == 1:
        axes = [axes]
    max_height = max(max(sum(record[2]), sum(record[3])) for record in picked)

    for i, (_, bar_reduction_pct, baseline_parts, sd_parts, _) in enumerate(picked):
        ax = axes[i]
        for x, parts in zip([0.3, 1.3], [baseline_parts, sd_parts]):
            tool, prefill, decode = parts
            ax.bar(x, tool, width=0.6, color=C_TOOL, edgecolor='white', linewidth=0.4)
            ax.bar(x, prefill, width=0.6, bottom=tool,
                   color=C_PREF, edgecolor='white', linewidth=0.4)
            ax.bar(x, decode, width=0.6, bottom=tool + prefill,
                   color=C_DEC, edgecolor='white', linewidth=0.4)

        ax.set_xticks([0.3, 1.3])
        ax.set_xticklabels(['BL', SYSNAME_SHORT], fontsize=7)
        ax.set_xlim(0, 1.6)
        ax.set_ylim(0, max_height * 1.08)
        ax.set_title(f'Req {i + 1}', fontsize=8)
        if i == 0:
            ax.set_ylabel('Latency (s)')
        ax.annotate(f'-{bar_reduction_pct:.1f}%',
                    xy=(1.3, sum(sd_parts)), xytext=(0, 2),
                    textcoords='offset points', ha='center', va='bottom', fontsize=7,
                    fontweight='bold', color='green')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, axis='y', alpha=0.3)

    legend = [
        Patch(facecolor=C_TOOL, edgecolor='gray', label='Tool Time'),
        Patch(facecolor=C_PREF, edgecolor='gray', label='Prefill'),
        Patch(facecolor=C_DEC, edgecolor='gray', label='Decode'),
    ]
    fig.legend(handles=legend, fontsize=8, frameon=False, ncol=3,
               loc='lower center', bbox_to_anchor=(0.5, 0.01))
    fig.suptitle(title, fontsize=11, fontweight='semibold', y=0.97)
    fig.subplots_adjust(left=0.08, right=0.99, top=0.82, bottom=0.25, wspace=0.25)

    save_path = os.path.join(PLOT_DIR, save_name)
    plt.savefig(save_path, bbox_inches='tight')
    print(f'Saved to {save_path}')
    plt.show()


plot_request_breakdown(
    1.5,
    TAG_BL,
    TAG_SD,
    'BFCL v4 Web Search — Request breakdown',
    'bfcl_breakdown.pdf',
)

In [ ]:
# BFCL disaggregated: median FTR / E2E, baseline vs Sutradhara (DS+KV)
TAG_DISAGG_BL, TAG_DISAGG_SD = 'disagg', 'disagg_ds_kv'

print(f"{'':>5}|{'FTR p50':>13}{'':>10}|{'E2E p50':>13}{'':>10}|")
print(f'-'*55)
print(f"{'QPS':>5}|" + f"{'BL':>7}{'SD':>7}{'gain':>9}|" * 2)
for q in QPS_LIST:
    if not (have(q, TAG_DISAGG_BL) and have(q, TAG_DISAGG_SD)):
        continue
    bl, sd = req_rows(q, TAG_DISAGG_BL), req_rows(q, TAG_DISAGG_SD)
    bf, sf = pctls(bl, 'first_token_render_ms'), pctls(sd, 'first_token_render_ms')
    be, se = pctls(bl, 'e2e_time_ms'), pctls(sd, 'e2e_time_ms')
    line = f"{q:>5}|"
    for b, s in ((bf, sf), (be, se)):
        line += f"{b['p50']:>7.2f}{s['p50']:>7.2f}{(b['p50']-s['p50'])/b['p50']*100:>+8.1f}%|"
    print(line)